# Milestone 3: Evaluation and Refinement

Objective:
Evaluate recommendation model performance and refine the algorithm.

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
interaction_df = pd.read_csv(
    "../Milestone_1/cleaned_data/cleaned_interactions.csv"
)

interaction_df.head()

,user_id,product_id,quantity
0,12346,23166,74215
1,12347,16008,24
2,12347,17021,36
3,12347,20665,6
4,12347,20719,40


In [3]:
# Shuffle data
interaction_df = interaction_df.sample(frac=1, random_state=42)

# Split 80% train, 20% test
train_size = int(0.8 * len(interaction_df))
train_data = interaction_df[:train_size]
test_data = interaction_df[train_size:]

In [4]:
train_matrix = train_data.pivot_table(
    index='user_id',
    columns='product_id',
    values='quantity',
    fill_value=0
)

In [5]:
user_similarity = cosine_similarity(train_matrix)

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=train_matrix.index,
    columns=train_matrix.index
)

In [6]:
def recommend_products(user_id, top_n=5):
    if user_id not in train_matrix.index:
        return []

    similar_users = user_similarity_df[user_id].sort_values(ascending=False)[1:6]
    scores = train_matrix.loc[similar_users.index].T.dot(similar_users)
    scores = scores[train_matrix.loc[user_id] == 0]

    return scores.sort_values(ascending=False).head(top_n).index.tolist()

In [7]:
def precision_recall_f1(user_id, top_n=5):
    # Actual products from test set
    actual_items = test_data[test_data['user_id'] == user_id]['product_id'].tolist()
    if not actual_items:
        return None

    # Recommended products
    recommended_items = recommend_products(user_id, top_n)

    # True positives
    tp = len(set(actual_items) & set(recommended_items))

    precision = tp / len(recommended_items) if recommended_items else 0
    recall = tp / len(actual_items) if actual_items else 0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0

    return precision, recall, f1


In [8]:
results = []

for user in test_data['user_id'].unique()[:10]:
    metrics = precision_recall_f1(user)
    if metrics:
        results.append(metrics)

results_df = pd.DataFrame(
    results,
    columns=['Precision', 'Recall', 'F1-score']
)

results_df.mean()


Precision    0.120000
Recall       0.015159
F1-score     0.026108
dtype: float64